# 06 Multi-Head Attention 多头注意力机制

前面我们已经学过单个 Self-Attention 的核心流程：

- $QK^{\top}$ 算注意力分数
- Softmax 把分数变成权重
- 权重乘 $V$ 得到融合上下文的新表示

这一节学习 Multi-Head Attention，也就是多头注意力机制。

先给一句核心直觉：

```text
一个 attention head 像一种观察关系的视角。
Multi-Head Attention 就是让模型同时用多个视角理解同一段输入。
```

这一节暂时不写代码。

目标是先把“为什么要多头”和“多头的形状怎么变”讲明白。

## 1. 为什么一个 head 可能不够

一段文本里，token 之间的关系通常不止一种。

例如这句话：

```text
聪明 勤奋 的 学生 通过了 考试，因为 她 准备 得 很 充分。
```

里面至少有几种关系：

```text
聪明、勤奋 -> 修饰 学生
她 -> 可能指 学生
准备充分 -> 解释 通过考试
考试 -> 和 通过了 构成事件关系
```

如果只有一个 attention head，它当然也能学习一些关系。

但一个头的注意力权重只有一张注意力表。

这一张表很难同时清楚表达多种不同关系。

所以很自然地想到：

```text
能不能让模型同时画多张注意力表？
每张表关注一种不同关系。
```

## 2. 一个 head 可以理解成一种关系探测器

回到 3Blue1Brown 的讲法。

他会先让你把一个 attention head 想成在学习某一种具体关系。

例如 head 1 可能学：

```text
形容词如何更新名词。
```

那么在句子：

```text
fluffy blue creature
```

这个 head 可能让 creature 更多关注 fluffy 和 blue。

再比如 head 2 可能学：

```text
代词如何找到指代对象。
```

那么在句子：

```text
小红 明天 要 考试，因为 她 很 紧张
```

这个 head 可能让“她”更多关注“小红”。

所以可以先这样理解：

一个 head = 一套独立的 $Q$、$K$、$V$ = 一张注意力图 = 一种可能的关系视角。

## 3. Multi-Head 的核心思想

Multi-Head Attention 的核心思想很朴素：

- 不要只算一套 $Q$、$K$、$V$。
- 而是并行算多套 $Q$、$K$、$V$。
- 每一套形成一个 head。
- 每个 head 各自做一次 Self-Attention。
- 最后把多个 head 的结果合并起来。

如果有 4 个 heads，就相当于：

```text
head 1：看一种关系
head 2：看另一种关系
head 3：再看一种关系
head 4：再看一种关系
```

它不是把同一个结果重复 4 遍。

每个 head 有自己的参数，因此可以学习不同的表示空间和不同的注意力模式。

## 4. 先复习单头的形状

先不考虑 batch。

假设输入是：

$$
\begin{aligned}
X &: N \times D
\end{aligned}
$$

单个 Self-Attention 会生成：

$$
\begin{aligned}
Q &: N \times d_{k} \\
K &: N \times d_{k} \\
V &: N \times d_{v}
\end{aligned}
$$

然后：

$$
\begin{aligned}
QK^{\top} &: (N \times d_{k}) \cdot (d_{k} \times N) \rightarrow N \times N \\
\text{Softmax 后} &: N \times N \\
\text{乘 V} &: (N \times N) \cdot (N \times d_{v}) \rightarrow N \times d_{v}
\end{aligned}
$$

单个 head 的输出是：

$$
\begin{aligned}
N \times d_{v}
\end{aligned}
$$

多头就是把这件事并行做多次。

## 5. 多头不是让维度无限变大

初学 Multi-Head 时，一个常见误解是：

```text
如果有 8 个头，是不是输出维度就变成原来的 8 倍？
```

通常不是这样。

常见做法是把总维度分给多个头。

比如模型总维度是：

$$
\begin{aligned}
D &= 512
\end{aligned}
$$

如果有：

$$
\begin{aligned}
h &= \text{8 个 heads}
\end{aligned}
$$

那么每个 head 可以使用：

$$
\begin{aligned}
d_{\mathrm{head}} &= \frac{512}{8} = 64
\end{aligned}
$$

每个 head 的维度变小。

多个 head 拼接回来后，总维度仍然是 512。

所以：

```text
Multi-Head 不是简单扩大输出维度。
而是把表示空间切成多个子空间，并行学习不同关系。
```

## 6. 用一个小例子看多头维度

为了让形状更清楚，我们用小数字。

假设：

$$
\begin{aligned}
N &= 4 \quad \text{一句话有 4 个 token} \\
D &= 8 \quad \text{每个 token 用 8 维表示} \\
h &= 2 \quad \text{使用 2 个 heads} \\
d_{\mathrm{head}} &= 4 \quad \text{每个 head 使用 4 维}
\end{aligned}
$$

输入是：

$$
\begin{aligned}
X &: 4 \times 8
\end{aligned}
$$

head 1 会得到：

$$
\begin{aligned}
Q_{1} &: 4 \times 4 \\
K_{1} &: 4 \times 4 \\
V_{1} &: 4 \times 4
\end{aligned}
$$

head 2 也会得到：

$$
\begin{aligned}
Q_{2} &: 4 \times 4 \\
K_{2} &: 4 \times 4 \\
V_{2} &: 4 \times 4
\end{aligned}
$$

每个 head 都独立做 Self-Attention。

所以：

$$
\begin{aligned}
\text{head 1 输出} &: 4 \times 4 \\
\text{head 2 输出} &: 4 \times 4
\end{aligned}
$$

## 7. 每个 head 都有自己的 $Q$、$K$、$V$ 参数

多头之所以能学不同关系，是因为每个 head 有自己的参数。

比如 head 1 使用：

- $W_{Q}^{(1)}$
- $W_{K}^{(1)}$
- $W_{V}^{(1)}$

head 2 使用：

- $W_{Q}^{(2)}$
- $W_{K}^{(2)}$
- $W_{V}^{(2)}$

它们不是同一套参数。

所以同一个输入 $X$，经过不同 head 的变换后，会进入不同的表示空间。

可以理解成：

```text
head 1 戴一副眼镜看句子。
head 2 戴另一副眼镜看句子。
```

同一句话，在不同眼镜下，看到的关系可能不一样。

## 8. head 1 怎么算

先看 head 1。

它有自己的：

$$
\begin{aligned}
Q_{1} &: 4 \times 4 \\
K_{1} &: 4 \times 4 \\
V_{1} &: 4 \times 4
\end{aligned}
$$

计算注意力分数：

$$
\begin{aligned}
Q_{1} K_{1}^{\top} &: (4 \times 4) \cdot (4 \times 4) \rightarrow 4 \times 4
\end{aligned}
$$

Softmax 后仍然是：

$$
\begin{aligned}
4 \times 4
\end{aligned}
$$

再乘 $V$：

$$
\begin{aligned}
(4 \times 4) \cdot (4 \times 4) \rightarrow 4 \times 4
\end{aligned}
$$

所以 head 1 输出：

$$
\begin{aligned}
O_{1} &: 4 \times 4
\end{aligned}
$$

这表示每个 token 在 head 1 这个视角下，得到一个 4 维的新表示。

## 9. head 2 怎么算

head 2 的计算形式和 head 1 一样。

但是它用的是另一套参数。

所以：

$$
\begin{aligned}
Q_{2} &: 4 \times 4 \\
K_{2} &: 4 \times 4 \\
V_{2} &: 4 \times 4
\end{aligned}
$$

计算分数：

$$
\begin{aligned}
Q_{2} K_{2}^{\top} &: (4 \times 4) \cdot (4 \times 4) \rightarrow 4 \times 4
\end{aligned}
$$

Softmax：

$$
\begin{aligned}
4 \times 4 \rightarrow 4 \times 4
\end{aligned}
$$

乘 $V$：

$$
\begin{aligned}
(4 \times 4) \cdot (4 \times 4) \rightarrow 4 \times 4
\end{aligned}
$$

head 2 输出：

$$
\begin{aligned}
O_{2} &: 4 \times 4
\end{aligned}
$$

虽然形状一样，但 head 1 和 head 2 的注意力表可能不同，输出内容也可能不同。

## 10. 把多个 head 的结果拼起来

现在我们有两个 head 的输出：

$$
\begin{aligned}
O_{1} &: 4 \times 4 \\
O_{2} &: 4 \times 4
\end{aligned}
$$

接下来把它们在最后一维拼接起来。

也就是每个 token 的两个 4 维结果，拼成一个 8 维结果。

形状变化是：

$$
\begin{aligned}
\operatorname{Concat}(O_{1}, O_{2}) &: 4 \times 8
\end{aligned}
$$

这里的 8 来自：

$$
\begin{aligned}
\text{2 个 heads} \times \text{每个 head 4 维} &= \text{8 维}
\end{aligned}
$$

所以多头拼接后又回到了模型总维度：

$$
\begin{aligned}
4 \times 8
\end{aligned}
$$

这就是为什么多头之后形状通常还能和输入保持一致。

## 11. 为什么拼接后还要一个输出线性层

Multi-Head Attention 通常不会在拼接后直接结束。

还会接一个输出线性变换，常写作：

$$
W_O
$$

如果拼接后的结果是：

$$
\begin{aligned}
4 \times 8
\end{aligned}
$$

$W_O$ 可以是：

$$
\begin{aligned}
8 \times 8
\end{aligned}
$$

那么输出仍然是：

$$
\begin{aligned}
(4 \times 8) \cdot (8 \times 8) \rightarrow 4 \times 8
\end{aligned}
$$

这个输出线性层的作用可以先理解成：

```text
把多个 head 拼出来的信息再混合整理一次。
```

多个 head 像多个人分别写了观察报告。

$W_O$ 像是把这些报告重新整理成统一格式，交给下一层继续处理。

## 12. 把小例子的完整路线串起来

现在把小例子完整串起来。

$$
\begin{aligned}
\text{输入 }X &: 4 \times 8 \\
\text{head 1} \\
Q_{1} &: 4 \times 4 \\
K_{1} &: 4 \times 4 \\
V_{1} &: 4 \times 4 \\
\text{Attention 输出 }O_{1} &: 4 \times 4 \\
\text{head 2} \\
Q_{2} &: 4 \times 4 \\
K_{2} &: 4 \times 4 \\
V_{2} &: 4 \times 4 \\
\text{Attention 输出 }O_{2} &: 4 \times 4 \\
\text{拼接} &: \operatorname{Concat}(O_{1}, O_{2}):4 \times 8 \\
\text{输出线性层} &: (4 \times 8) \cdot (8 \times 8) \rightarrow 4 \times 8 \\
\text{最终输出} &: 4 \times 8
\end{aligned}
$$

这条线最重要。

它说明 Multi-Head Attention 虽然内部拆成多个头，但最终仍然输出每个 token 的新表示。

## 13. 推广到一般符号

现在把具体数字换成一般符号。

不考虑 batch 时：

$$
\begin{aligned}
\text{输入 }X &: N \times D \\
\text{head 数量} &: h \\
\text{每个 head 的维度} &: d_{\mathrm{head}} \\
\text{通常 }D &= h \times d_{\mathrm{head}}
\end{aligned}
$$

每个 head 的输出是：

$$
\begin{aligned}
N \times d_{\mathrm{head}}
\end{aligned}
$$

$h$ 个 heads 拼接后：

$$
\begin{aligned}
N \times (h \times d_{\mathrm{head}})
\end{aligned}
$$

如果 $D=h\times d_{\mathrm{head}}$，那么就是：

$$
\begin{aligned}
N \times D
\end{aligned}
$$

再经过输出线性层后，通常仍然保持：

$$
\begin{aligned}
N \times D
\end{aligned}
$$

所以 Transformer 里很多层可以堆起来，因为每层输入输出形状可以保持一致。

## 14. 加上 batch 后的形状

真实训练时通常有 batch。

输入是：

$$
\begin{aligned}
X &: B \times N \times D
\end{aligned}
$$

生成 $Q$、$K$、$V$ 后，常见实现会整理成：

$$
\begin{aligned}
Q &: B \times h \times N \times d_{\mathrm{head}} \\
K &: B \times h \times N \times d_{\mathrm{head}} \\
V &: B \times h \times N \times d_{\mathrm{head}}
\end{aligned}
$$

这里多了一个维度：

$h$：head 数量

每个 batch 样本、每个 head 内部，都有自己的注意力表：

$$
\begin{aligned}
\text{注意力分数} &: B \times h \times N \times N
\end{aligned}
$$

乘 $V$ 后：

$$
\begin{aligned}
\text{每个 head 输出} &: B \times h \times N \times d_{\mathrm{head}}
\end{aligned}
$$

再把 heads 拼回去：

$$
\begin{aligned}
B \times N \times D
\end{aligned}
$$

这就是多头注意力在实际模型里最常见的形状路线。

## 15. 为什么要把 $D$ 拆成 $h$ 个 $d_{\mathrm{head}}$

现在解释一个很重要的设计。

假设总维度是 $D$。

如果每个 head 都使用完整的 $D$ 维，那么 $h$ 个 heads 的计算量和参数量会变得很大。

常见做法是让每个 head 使用较小的子空间：

$$
d_{\mathrm{head}}=\frac{D}{h}
$$

这样做有两个好处。

第一，多个 head 可以从不同子空间观察关系。

第二，拼接回来后总维度仍然接近原来的 $D$，方便接到下一层。

可以这样理解：

```text
不是每个 head 都拿完整大脑思考所有问题。
而是把模型维度分成多个小工作区，让多个 head 并行处理不同关系。
```

## 16. 多头到底能学到什么关系

理论上，不同 head 可以学习不同类型的关系。

例如：

```text
head 1：形容词修饰名词
head 2：代词指代人物
head 3：动词关联主语
head 4：否定词影响后面动词
head 5：时间词影响事件理解
head 6：标点或分隔符组织句子结构
```

不过这里要谨慎。

真实模型里的 head 不一定都能被人类清楚解释成某一种语法关系。

有些 head 可能比较容易解释。

有些 head 可能学到的是更混合、更抽象的模式。

所以更稳妥的说法是：

```text
Multi-Head 给模型提供多个并行的关系建模通道。
这些通道有机会学习不同类型的信息交互。
```

## 17. 多头和单头的核心区别

单头 Attention：

- 一套 $Q$、$K$、$V$
- 一张注意力表
- 一个关系视角

多头 Attention：

- 多套 $Q$、$K$、$V$
- 多张注意力表
- 多个关系视角
- 拼接后再统一整理

如果用一句话概括：

```text
单头是在一个空间里看关系。
多头是在多个子空间里并行看关系。
```

## 18. 多头注意力的完整公式怎么读

Multi-Head Attention 常写成：

$$
\operatorname{MultiHead}(Q,K,V)=\operatorname{Concat}(\mathrm{head}_{1},\mathrm{head}_{2},\dots,\mathrm{head}_{h})W_O
$$

其中每个 head 是：

$$
\mathrm{head}_{i}=\operatorname{Attention}(QW_Q^i,KW_K^i,VW_V^i)
$$

第一次看这个公式，重点不要放在背符号。

要这样读：

- 第 i 个 head 有自己的 $Q$、$K$、$V$ 变换。
- 它独立做一次 Attention。
- 所有 head 的结果拼接。
- 最后用 $W_{O}$ 再整理一次。

公式只是把这几句话压缩成数学写法。

## 19. 为什么拼接而不是相加

多个 head 得到的是不同视角的信息。

如果直接相加，很多细节会混在一起。

拼接的好处是先保留每个 head 的输出。

例如：

```text
head 1 输出前 4 维
head 2 输出后 4 维
```

拼接后，模型还能知道哪些信息来自哪个 head 的子空间。

然后再通过 $W_O$ 统一混合。

所以顺序是：

```text
先保留各头信息
再学习如何融合
```

这比一开始就粗暴相加更灵活。

## 20. Multi-Head 和 3Blue1Brown 视角怎么接上

3Blue1Brown 讲一个 head 时，会先用“形容词更新名词”建立直觉。

到了 Multi-Head，可以把这个直觉扩展：

```text
不是只有一种信息更新方式。
不同 head 可以尝试不同的更新方式。
```

比如同一句话里：

```text
聪明 勤奋 的 学生 通过了 考试，因为 她 准备 得 很 充分。
```

某个 head 可能让“学生”关注“聪明、勤奋”。

某个 head 可能让“她”关注“学生”。

某个 head 可能让“通过了”关注“考试”。

某个 head 可能让“通过了考试”关联“准备充分”。

所以 Multi-Head 更像是：

```text
多种上下文改写方式同时发生。
```

## 21. 多头注意力的代价

Multi-Head 更灵活，但也有代价。

每个 head 都要计算注意力表。

如果输入长度是 $N$，每个 head 都会产生一个：

$$
N\times N
$$

的注意力表。

如果有 $h$ 个 heads，那么注意力表数量就是 $h$ 张。

加上 batch 后，注意力分数形状是：

$$
\begin{aligned}
B \times h \times N \times N
\end{aligned}
$$

所以序列长度 $N$ 很大时，计算和显存压力会明显增加。

这也是为什么长文本 Transformer 会特别关注注意力计算效率。

## 22. 常见误解 1：head 越多一定越好

head 多，确实给了模型更多关系建模通道。

但不是越多越好。

因为总维度 $D$ 通常固定。

head 数量 $h$ 越多，每个 head 分到的维度 $d_{\mathrm{head}}$ 就越小。

如果每个 head 的维度太小，它表达复杂关系的能力也可能受限制。

所以 head 数量是一个设计平衡：

- 更多 head：更多关系视角。
- 更小 $d_{\mathrm{head}}$：每个视角可用的信息维度变少。

模型设计里通常会根据总维度、任务规模和计算资源选择合适的 head 数。

## 23. 常见误解 2：每个 head 都一定有人类可解释含义

前面我们用“形容词修饰名词”“代词指代人物”来解释 head。

这些例子很适合理解。

但真实模型里，不要期待每个 head 都能被清楚翻译成人类语言规则。

原因是：

```text
模型学到的是高维向量里的统计模式。
这些模式可能和人类语法规则有重叠。
但也可能更混合、更抽象。
```

所以我们用这些例子建立直觉，但不要把它当成每个 head 的硬性解释。

## 24. 常见误解 3：Multi-Head 只是多个模型投票

Multi-Head Attention 不是多个完整模型在投票。

它是在同一个层内部，把注意力计算拆成多个 head 并行进行。

这些 head 的输出会拼接，然后经过 $W_O$ 混合。

所以它不是：

```text
多个模型分别给答案，然后投票。
```

而是：

```text
同一个模型里，多个注意力视角并行提取上下文信息，然后合并成一个表示。
```

## 25. 和 Transformer Encoder 的关系

Multi-Head Attention 是 Transformer Encoder 里的核心组件之一。

一个典型 Encoder 层里，通常会有：

```text
Multi-Head Self-Attention
Add & Norm
Feed-Forward Network
Add & Norm
```

这一节只讲了第一个部分：

```text
Multi-Head Self-Attention
```

后面还要继续学习：

```text
为什么需要残差连接
为什么需要 LayerNorm
Feed-Forward Network 做什么
位置编码怎么加入
```

现在先把 Multi-Head 的主线掌握好。

## 26. 本节小结

这一节先记住：

1. 一个 attention head 可以理解成一种关系视角。
2. Multi-Head Attention 让多个 head 并行工作。
3. 每个 head 有自己的 $Q$、$K$、$V$ 参数，可以学不同注意力模式。
4. 常见做法是把总维度 $D$ 拆成 $h$ 个 $d_{\mathrm{head}}$。
5. 每个 head 输出 $N \times d_{\mathrm{head}}$。
6. 多个 head 拼接后得到 $N \times D$。
7. 拼接后通常再经过输出线性层 $W_O$ 做统一整理。
8. 加上 batch 后，注意力分数常见形状是 $B \times h \times N \times N$。
9. head 多不一定永远更好，需要和每个 head 的维度、计算量做平衡。
10. Multi-Head Attention 是 Transformer Encoder 的核心组件之一。

## 27. 自测问题

1. 为什么一个 attention head 可能不够？
2. 一个 head 可以先理解成什么？
3. Multi-Head Attention 的核心思想是什么？
4. 如果 $D = 512$，$h = 8$，那么每个 head 的维度通常是多少？
5. 为什么 Multi-Head 通常不是把输出维度扩大 $h$ 倍？
6. 在 $N=4, D=8, h=2$ 的例子里，每个 head 输出什么形状？
7. 两个 $4 \times 4$ 的 head 输出拼接后是什么形状？
8. 输出线性层 $W_O$ 的作用是什么？
9. 加上 batch 后，$Q$、$K$、$V$ 常见形状是什么？
10. 为什么注意力分数形状会是 $B \times h \times N \times N$？
11. 为什么 head 越多不一定越好？
12. 为什么不能要求每个 head 都有清晰的人类可解释含义？
13. Multi-Head Attention 和多个模型投票有什么区别？
14. Multi-Head Attention 在 Transformer Encoder 里处于什么位置？